# Data preprocessing

## Import required libraries

In [2]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

## Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Organize paths

In [3]:
DATA_PATH = "../VQA Dataset/"
questions_file = DATA_PATH + "questions/v2_OpenEnded_mscoco_val2014_questions.json"
annotations_file = DATA_PATH + "annotations/v2_mscoco_val2014_annotations.json"
images_path = DATA_PATH + "images/val2014/"

# Check if the directory exists before trying to list its contents
if os.path.isdir(images_path):
    image_files = os.listdir(images_path)
    print(f"Found {len(image_files)} image files in {images_path}")
else:
    print(f"Error: Image directory not found at {images_path}. Please verify the path.")
    image_files = [] # Initialize as empty list to prevent further errors

Found 40504 image files in ../VQA Dataset/images/val2014/


In [4]:
df = pd.read_csv("../visual-question-answering/VQA_merged_final.csv")

In [5]:
df.head()

,image_id,tokens,question_tensor,lstm_embedding,answer_label,soft_score
0,393225,"['is', 'this', 'a', 'creamy', 'soup', '<PAD>',...","[3, 6, 10, 1, 1039, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0.008301198482513428, 0.01019968930631876, 0....",1,0.3333
1,393225,"['is', 'this', 'rice', 'noodle', 'soup', '<PAD...","[3, 6, 1314, 5489, 1039, 0, 0, 0, 0, 0, 0, 0, ...","[0.009467930532991886, 0.010444258339703083, 0...",0,0.3333
2,240301,"['why', 'is', 'there', 'a', 'gap', 'between', ...","[70, 3, 14, 10, 1, 448, 2, 679, 97, 88, 0, 0, ...","[-0.0008158856071531773, 0.016982674598693848,...",0,0.6667
3,240301,"['is', 'it', 'daylight', 'in', 'this', 'pictur...","[3, 22, 1824, 7, 6, 18, 0, 0, 0, 0, 0, 0, 0, 0]","[0.010015132836997509, 0.00946513470262289, 0....",0,0.6667
4,131089,"['did', 'the', 'batter', 'hit', 'the', 'ball',...","[144, 2, 359, 353, 2, 148, 0, 0, 0, 0, 0, 0, 0...","[0.010038641281425953, 0.009127109311521053, 0...",0,0.6667


In [ ]:
# =========================
# VQA TRAINING PIPELINE
# =========================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd
import ast

# =========================
# 1. LOAD DATA
# =========================

df = pd.read_csv("VQA_merged_final.csv")

# embeddings images déjà calculés
image_embeddings = torch.load("../images_feats.pt", weights_only=False)

# vocab pour LSTM
import json
with open("word_vocab.json") as f:
    word2idx = json.load(f)

PAD_IDX = word2idx.get("<PAD>", 0)


# =========================
# 2. DATASET
# =========================

class VQADataset(Dataset):
    def __init__(self, df, image_embeddings):
        self.image_embeddings = image_embeddings

        self.image_ids = df["image_id"].values
        self.labels = torch.tensor(df["answer_label"].values, dtype=torch.long)

        # conversion 
        self.questions = [
            torch.tensor(ast.literal_eval(q), dtype=torch.long)
            if isinstance(q, str) else torch.tensor(q, dtype=torch.long)
            for q in df["question_tensor"].values
        ]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_feat = self.image_embeddings[int(self.image_ids[idx])]
        q_tensor = self.questions[idx]
        label = self.labels[idx]
        return img_feat, q_tensor, label

# =========================
# 3. MODEL
# =========================

class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        return torch.cat([h[-2], h[-1]], dim=1)  # (B, 1024)

class VQAModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.q_encoder = LSTMEncoder(vocab_size)

        # projections
        self.img_proj = nn.Linear(2048, 512)
        self.q_proj   = nn.Linear(1024, 512)

        # classifier
        self.fc = nn.Linear(512, 2)
        self.dropout = nn.Dropout(0.3)

    def forward(self, img_feat, q_tensor):
        q_feat = self.q_encoder(q_tensor)

        img_feat = self.img_proj(img_feat)
        q_feat   = self.q_proj(q_feat)

        x = img_feat * q_feat   # element-wise interaction

        x = torch.relu(x)
        x = self.dropout(x)

        return self.fc(x)

# =========================
# 4. DATALOADER
# =========================

dataset = VQADataset(df, image_embeddings)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0  # important sur Mac / Colab
)

In [12]:
# =========================
# VQA TRAINING PIPELINE
# =========================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd
import ast

# =========================
# 1. LOAD DATA
# =========================

df = pd.read_csv("VQA_merged_final.csv")

# embeddings images déjà calculés
image_embeddings = torch.load("../images_feats.pt", weights_only=False)

# vocab pour LSTM
import json
with open("word_vocab.json") as f:
    word2idx = json.load(f)

PAD_IDX = word2idx.get("<PAD>", 0)


# =========================
# 2. DATASET
# =========================

class VQADataset(Dataset):
    def __init__(self, df, image_embeddings):
        self.image_embeddings = image_embeddings

        self.image_ids = df["image_id"].values
        self.labels = torch.tensor(df["answer_label"].values, dtype=torch.long)

        # conversion 
        self.questions = [
            torch.tensor(ast.literal_eval(q), dtype=torch.long)
            if isinstance(q, str) else torch.tensor(q, dtype=torch.long)
            for q in df["question_tensor"].values
        ]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_feat = self.image_embeddings[int(self.image_ids[idx])]
        q_tensor = self.questions[idx]
        label = self.labels[idx]
        return img_feat, q_tensor, label

# =========================
# 3. MODEL
# =========================

class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        return torch.cat([h[-2], h[-1]], dim=1)  # (B, 1024)

class VQAModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.q_encoder = LSTMEncoder(vocab_size)

        # projections
        self.img_proj = nn.Linear(2048, 512)
        self.q_proj   = nn.Linear(1024, 512)

        # classifier
        self.fc = nn.Linear(512, 2)
        self.dropout = nn.Dropout(0.3)

    def forward(self, img_feat, q_tensor):
        q_feat = self.q_encoder(q_tensor)

        img_feat = self.img_proj(img_feat)
        q_feat   = self.q_proj(q_feat)

        x = img_feat * q_feat   # element-wise interaction

        x = torch.relu(x)
        x = self.dropout(x)

        return self.fc(x)

# =========================
# 4. DATALOADER
# =========================

dataset = VQADataset(df, image_embeddings)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0  # important sur Mac / Colab
)
# =========================
# 5. TRAINING
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VQAModel(vocab_size=len(word2idx)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 25

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch}")

    for img_feat, q_tensor, labels in pbar:
        img_feat = img_feat.to(device).float()
        q_tensor = q_tensor.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(img_feat, q_tensor)
        loss = F.cross_entropy(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix({
            "loss": f"{loss.item():.3f}",
            "acc": f"{correct/total:.3f}"
        })

    print(f"Epoch {epoch} | Loss: {total_loss/total:.4f} | Acc: {correct/total:.4f}")

Epoch 0: 100%|██████████| 1263/1263 [01:30<00:00, 13.93it/s, loss=0.711, acc=0.532]


Epoch 0 | Loss: 0.6899 | Acc: 0.5325


Epoch 1: 100%|██████████| 1263/1263 [01:23<00:00, 15.07it/s, loss=0.701, acc=0.551]


Epoch 1 | Loss: 0.6837 | Acc: 0.5514


Epoch 2: 100%|██████████| 1263/1263 [01:22<00:00, 15.28it/s, loss=0.686, acc=0.563]


...

Epoch 12 | Loss: 0.5800 | Acc: 0.6810


Epoch 13: 100%|██████████| 1263/1263 [01:37<00:00, 12.99it/s, loss=0.506, acc=0.692]


Epoch 13 | Loss: 0.5651 | Acc: 0.6924


Epoch 14:  72%|███████▏  | 905/1263 [01:10<00:27, 12.91it/s, loss=0.506, acc=0.707]


KeyboardInterrupt: 

In [15]:
# =========================
# VQA TRAINING + VALIDATION
# =========================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd
import ast
from sklearn.model_selection import train_test_split

# =========================
# 1. LOAD DATA
# =========================

df = pd.read_csv("VQA_merged_final.csv")

image_embeddings = torch.load("../images_feats.pt", weights_only=False)

import json
with open("word_vocab.json") as f:
    word2idx = json.load(f)

PAD_IDX = word2idx.get("<PAD>", 0)

# =========================
# 2. TRAIN / VAL SPLIT
# =========================

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)

# =========================
# 3. DATASET
# =========================

class VQADataset(Dataset):
    def __init__(self, df, image_embeddings):
        self.image_embeddings = image_embeddings

        self.image_ids = df["image_id"].values
        self.labels = torch.tensor(df["answer_label"].values, dtype=torch.long)

        self.questions = [
            torch.tensor(ast.literal_eval(q), dtype=torch.long)
            if isinstance(q, str) else torch.tensor(q, dtype=torch.long)
            for q in df["question_tensor"].values
        ]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_feat = self.image_embeddings[int(self.image_ids[idx])].float()
        q_tensor = self.questions[idx]
        label = self.labels[idx]
        return img_feat, q_tensor, label

# =========================
# 4. MODEL
# =========================

class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        return torch.cat([h[-2], h[-1]], dim=1)

class VQAModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.q_encoder = LSTMEncoder(vocab_size)

        self.img_proj = nn.Linear(2048, 512)
        self.q_proj   = nn.Linear(1024, 512)

        self.fc = nn.Linear(512 * 3, 2)
        self.dropout = nn.Dropout(0.5)

    def forward(self, img_feat, q_tensor):
        q_feat = self.q_encoder(q_tensor)

        img_feat = self.img_proj(img_feat)
        q_feat   = self.q_proj(q_feat)
        img_feat = F.normalize(img_feat, dim=1)
        q_feat   = F.normalize(q_feat, dim=1)

        x = torch.cat([img_feat, q_feat, img_feat * q_feat], dim=1)

        x = torch.relu(x)
        x = self.dropout(x)

        return self.fc(x)

# =========================
# 5. DATALOADERS
# =========================

train_loader = DataLoader(VQADataset(train_df, image_embeddings),
                          batch_size=64, shuffle=True, num_workers=0)

val_loader = DataLoader(VQADataset(val_df, image_embeddings),
                        batch_size=64, shuffle=False, num_workers=0)

In [16]:

# =========================
# 6. TRAIN + VALID LOOP
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VQAModel(vocab_size=len(word2idx)).to(device)
for param in model.q_encoder.parameters():
    param.requires_grad = False
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)

EPOCHS = 25
best_val_loss = float("inf")
patience = 3
counter = 0

for epoch in range(EPOCHS):

    # ===== TRAIN =====
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [TRAIN]")

    for img_feat, q_tensor, labels in pbar:
        img_feat = img_feat.to(device)
        q_tensor = q_tensor.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(img_feat, q_tensor)
        loss = F.cross_entropy(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)

        preds = outputs.argmax(1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

        pbar.set_postfix({
            "loss": f"{loss.item():.3f}",
            "acc": f"{train_correct/train_total:.3f}"
        })

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ===== VALID =====
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for img_feat, q_tensor, labels in val_loader:
            img_feat = img_feat.to(device)
            q_tensor = q_tensor.to(device)
            labels = labels.to(device)

            outputs = model(img_feat, q_tensor)
            loss = F.cross_entropy(outputs, labels)

            val_loss += loss.item() * labels.size(0)

            preds = outputs.argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    # ===== EARLY STOPPING =====
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_vqa_model.pt")
        print("Best model saved")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

Epoch 0 [TRAIN]: 100%|██████████| 1074/1074 [00:35<00:00, 29.98it/s, loss=0.677, acc=0.522]



Epoch 0
Train Loss: 0.6917 | Train Acc: 0.5215
Val   Loss: 0.6896 | Val   Acc: 0.5379
Best model saved


Epoch 1 [TRAIN]: 100%|██████████| 1074/1074 [00:34<00:00, 30.91it/s, loss=0.685, acc=0.535]



Epoch 1
Train Loss: 0.6897 | Train Acc: 0.5347
Val   Loss: 0.6883 | Val   Acc: 0.5453
Best model saved


Epoch 2 [TRAIN]: 100%|██████████| 1074/1074 [00:30<00:00, 35.76it/s, loss=0.662, acc=0.541]


...


Epoch 7
Train Loss: 0.6872 | Train Acc: 0.5443
Val   Loss: 0.6867 | Val   Acc: 0.5499
Best model saved


Epoch 8 [TRAIN]: 100%|██████████| 1074/1074 [00:31<00:00, 34.21it/s, loss=0.672, acc=0.545]



Epoch 8
Train Loss: 0.6867 | Train Acc: 0.5446
Val   Loss: 0.6874 | Val   Acc: 0.5521


Epoch 9 [TRAIN]:  92%|█████████▏| 993/1074 [00:31<00:02, 31.43it/s, loss=0.695, acc=0.545]


KeyboardInterrupt: 

In [19]:
# =========================
# VQA WITH BERT + IMAGE FEATURES
# =========================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
import pandas as pd

# =========================
# LOAD DATA
# =========================

df = pd.read_csv("VQA_merged_final.csv")
image_embeddings = torch.load("../images_feats.pt", weights_only=False)

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)

# =========================
# BERT TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

MAX_LEN = 20

# =========================
# DATASET
# =========================

class VQADataset(Dataset):
    def __init__(self, df, image_embeddings):
        self.df = df.reset_index(drop=True)
        self.image_embeddings = image_embeddings

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        question = row["tokens"]
        if isinstance(question, str):
            question = " ".join(eval(question))

        encoding = tokenizer(
            question,
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        img_feat = self.image_embeddings[int(row["image_id"])].float()

        return (
            img_feat,
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(row["answer_label"], dtype=torch.long)
        )

# =========================
# MODEL
# =========================

class VQAModel(nn.Module):
    def __init__(self):
        super().__init__()

        # BERT
        self.bert = AutoModel.from_pretrained("distilbert-base-uncased")

        # Freeze BERT au début
        for p in self.bert.parameters():
            p.requires_grad = False

        # projections
        self.img_proj = nn.Linear(2048, 512)
        self.q_proj   = nn.Linear(768, 512)

        # classifier
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, img_feat, input_ids, attention_mask):
        q_feat = self.bert(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state[:,0]

        img_feat = F.normalize(self.img_proj(img_feat), dim=1)
        q_feat   = F.normalize(self.q_proj(q_feat), dim=1)

        x = img_feat * q_feat   # Hadamard

        return self.fc(x)

# =========================
# DATALOADER
# =========================

train_loader = DataLoader(VQADataset(train_df, image_embeddings),
                          batch_size=64, shuffle=True)

val_loader = DataLoader(VQADataset(val_df, image_embeddings),
                        batch_size=64, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [21]:
# =========================
# TRAINING (OPTIMIZED)
# =========================

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VQAModel().to(device)

# LR différencié (clé)
optimizer = torch.optim.Adam([
    {"params": model.bert.parameters(), "lr": 2e-5},
    {"params": model.img_proj.parameters(), "lr": 1e-4},
    {"params": model.q_proj.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-4},
])

EPOCHS = 10
best_val = 0

for epoch in range(EPOCHS):

    # ===== UNFREEZE BERT =====
    if epoch == 2:
        print("\n Unfreezing BERT\n")
        for p in model.bert.parameters():
            p.requires_grad = True

    # ===== TRAIN =====
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [TRAIN]")

    for img_feat, input_ids, attn_mask, labels in pbar:
        img_feat = img_feat.to(device)
        input_ids = input_ids.to(device)
        attn_mask = attn_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(img_feat, input_ids, attn_mask)
        loss = F.cross_entropy(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix({
            "loss": f"{loss.item():.3f}",
            "acc": f"{correct/total:.3f}"
        })

    train_loss = total_loss / total
    train_acc = correct / total

    # ===== VALID =====
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(val_loader, desc=f"Epoch {epoch} [VAL]")

    with torch.no_grad():
        for img_feat, input_ids, attn_mask, labels in pbar:
            img_feat = img_feat.to(device)
            input_ids = input_ids.to(device)
            attn_mask = attn_mask.to(device)
            labels = labels.to(device)

            outputs = model(img_feat, input_ids, attn_mask)
            loss = F.cross_entropy(outputs, labels)

            total_loss += loss.item() * labels.size(0)

            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({
                "loss": f"{loss.item():.3f}",
                "acc": f"{correct/total:.3f}"
            })

    val_loss = total_loss / total
    val_acc = correct / total

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    # ===== SAVE BEST =====
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), "best_vqa_bert.pt")
        print("✓ Best saved")

Epoch 0 [VAL]: 100%|██████████| 190/190 [00:22<00:00,  8.29it/s, loss=0.684, acc=0.531]



Epoch 0
Train Loss: 0.6920 | Train Acc: 0.5238
Val   Loss: 0.6902 | Val   Acc: 0.5309
✓ Best saved


Epoch 1 [VAL]: 100%|██████████| 190/190 [00:22<00:00,  8.37it/s, loss=0.694, acc=0.544]



Epoch 1
Train Loss: 0.6885 | Train Acc: 0.5416
Val   Loss: 0.6875 | Val   Acc: 0.5441
✓ Best saved

 Unfreezing BERT



Epoch 2 [VAL]: 100%|██████████| 190/190 [00:25<00:00,  7.34it/s, loss=0.707, acc=0.557]


...


Epoch 3
Train Loss: 0.6670 | Train Acc: 0.5914
Val   Loss: 0.6742 | Val   Acc: 0.5751
✓ Best saved


Epoch 4 [VAL]: 100%|██████████| 190/190 [00:24<00:00,  7.82it/s, loss=0.765, acc=0.573]



Epoch 4
Train Loss: 0.6465 | Train Acc: 0.6217
Val   Loss: 0.6796 | Val   Acc: 0.5725


Epoch 5 [TRAIN]:  72%|███████▏  | 776/1074 [06:48<02:37,  1.90it/s, loss=0.619, acc=0.652]


KeyboardInterrupt: 